# Simple Model Machine Learning

### Version 1

In [8]:
# Get Data
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

data = pd.read_csv('housing/housing.csv')
data = data.dropna()

# Split Data
X = data[['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity']]
y = data['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scalling on X_train
scaler = StandardScaler()
onh_encoder = OneHotEncoder(handle_unknown='ignore')

num_cols = ['housing_median_age', 'total_rooms', 'total_bedrooms', 
            'population', 'households', 'median_income']
cat_cols = ['ocean_proximity']

X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

X_train_cat = onh_encoder.fit_transform(X_train[cat_cols]) # sparse matrix -> perlu menggunakan toarray() untuk melihat isi sparse matrix (jika data kecil, jangan di coba untuk data besar)
X_test_cat = onh_encoder.transform(X_test[cat_cols])

X_train_num_df = pd.DataFrame(X_train_num, columns=num_cols)
X_train_cat_df = pd.DataFrame(X_train_cat.toarray(), columns=onh_encoder.get_feature_names_out(cat_cols))
X_train_prepared = pd.concat([X_train_num_df, X_train_cat_df], axis=1)

X_test_num_df = pd.DataFrame(X_test_num, columns=num_cols)
X_test_cat_df = pd.DataFrame(X_test_cat.toarray(), columns=onh_encoder.get_feature_names_out(cat_cols))
X_test_prepared = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

# Train Data
model = LinearRegression()
model.fit(X_train_prepared, y_train)

# Test Data
y_pred = model.predict(X_test_prepared)

# Check Accuracy Data
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('RMSE:', rmse)



RMSE: 70322.00943952362


### Version 2

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer


data = pd.read_csv('housing/housing.csv')

train_set, test_set = train_test_split(data, test_size=0.2, random_state=42)
train_features = train_set.drop('median_house_value', axis=1)
train_labels = train_set['median_house_value'].copy()

X_test = test_set.drop('median_house_value', axis=1)
y_test = test_set['median_house_value'].copy()

num_cols = ['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
cat_cols = ['ocean_proximity']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

column_transform = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])


full_pipeline = Pipeline([
    ('preprocessing', column_transform),
    ('model', LinearRegression())
])

full_pipeline.fit(train_features, train_labels)

y_pred = full_pipeline.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(rmse)

70925.36950193123


### Version 3

In [ ]:
data = pd.read_csv('housing/housing.csv')

X = data.drop('median_house_value', axis=1)
y = data['median_house_value']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

num_cols = ['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
cat_cols = ['ocean_proximity']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

column_transform = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

full_pipeline = Pipeline([
    ('preprocessing', column_transform),
    ('model', LinearRegression())
])

full_pipeline.fit(X_train, y_train)

y_train_pred = full_pipeline.predict(X_train)
y_val_pred = full_pipeline.predict(X_val)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

y_test_pred  = full_pipeline.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print('Train:', train_rmse)
print('Val:',val_rmse)
print('Test:', test_rmse)


Train: 69905.62945452821
Val: 71645.4459624205
Test: 67908.37884842727
